In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import json

model_id = "Zual/MPropositioneur-V2"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")

texte = "The Falkland Islands (/ˈfɔː(l)klənd, ˈfɒlk-/;[8] Spanish: Islas Malvinas [ˈislas malˈβinas]), commonly referred to as the Falklands, is an archipelago in the South Atlantic Ocean on the Patagonian Shelf. The principal islands are about 300 mi (500 km) east of South America's southern Patagonian coast and 752 mi (1,210 km) from Cape Dubouzet at the northern tip of the Antarctic Peninsula, at a latitude of about 52°S. The archipelago, with an area of 4,700 sq mi (12,000 km2), comprises East Falkland, West Falkland, and 776 smaller islands. As a British Overseas Territory, the Falklands have internal self-governance, while the United Kingdom takes responsibility for their defence and foreign affairs. The capital and largest settlement is Stanley on East Falkland."

prompt = f"<|im_start|>user\nAtomize: {texte}<|im_end|>\n<|im_start|>assistant\n"
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=8192).to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=2048, do_sample=False)

generated_ids = outputs[0][inputs.input_ids.shape[1]:]
result = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

# La sortie est une liste JSON : ["p1", "p2", ...]
propositions = json.loads(result)
for p in propositions:
    print(f"• {p}")

Loading weights: 100%|██████████| 310/310 [00:01<00:00, 250.36it/s]


• The Falkland Islands are an archipelago in the South Atlantic Ocean.
• The Falkland Islands are located on the Patagonian Shelf.
• The Falkland Islands are commonly referred to as the Falklands.
• The Falkland Islands are also known as the Islas Malvinas.
• The Falkland Islands are a British Overseas Territory.
• The Falkland Islands have internal self-governance as a British Overseas Territory.
• The United Kingdom takes responsibility for the Falkland Islands' defence and foreign affairs.
• The capital of the Falkland Islands is Stanley on East Falkland.
• The largest settlement in the Falkland Islands is Stanley on East Falkland.
